## Lesson 2

In [1]:
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

139

In [3]:
documents = documents_llm

In [4]:
doc = documents[1]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

977bf7786c
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.


In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

response

ParsedResponse[TypeVar](id='resp_0738fd359b8a8117006a747264b2b8819fbab7809ceb4e0447', created_at=1786016356.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_0738fd359b8a8117006a7472658660819fa92415e754bcd00b', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"questions":["Is it still possible to join the course if I found it late?","Can new students enroll after the course has already started?","If I join now, will I still be eligible for a certificate?","Do I need to submit the project before submissions close to get the certificate?","What’s the deadline for project submission if I want the course certificate?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['Is it still possible to join the course if I found it late?', 'Can new students enroll after the course has already started?', 'If I join now, will I still be eligible 

In [12]:
result = response.output_parsed

print(result)

questions=['Is it still possible to join the course if I found it late?', 'Can new students enroll after the course has already started?', 'If I join now, will I still be eligible for a certificate?', 'Do I need to submit the project before submissions close to get the certificate?', 'What’s the deadline for project submission if I want the course certificate?']


In [13]:
result.questions

['Is it still possible to join the course if I found it late?',
 'Can new students enroll after the course has already started?',
 'If I join now, will I still be eligible for a certificate?',
 'Do I need to submit the project before submissions close to get the certificate?',
 'What’s the deadline for project submission if I want the course certificate?']

In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    client=openai_client,
    instructions=data_gen_instructions,
    user_prompt=user_prompt,
    output_type=Questions,
)

result.questions

['I’m new to this course—can I still join even if I found it late?',
 'If I start the course now, is it still possible to get a certificate?',
 'What’s the deadline for the final project if I want the certificate?',
 'Can I submit my project after the course is no longer accepting submissions and still get certified?',
 'If I join late, do I have to finish the project before submissions close to be eligible for the certificate?']

In [16]:
usage.input_tokens, usage.output_tokens

(207, 104)

In [17]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000468, 'total_cost': 0.00062325}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I’m new to this course—can I still join even if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, is it still possible to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for the final project if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I submit my project after the course is no longer accepting submissions and still get certified?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, do I have to finish the project before submissions close to be eligible for the certificate?',
  'document': '74eb249bbf'}]

## Lesson 3

In [20]:
import pandas as pd

In [22]:
pd.DataFrame(records)

,question,document
0,I’m new to this course—can I still join even i...,74eb249bbf
1,"If I start the course now, is it still possibl...",74eb249bbf
2,What’s the deadline for the final project if I...,74eb249bbf
3,Can I submit my project after the course is no...,74eb249bbf
4,"If I join late, do I have to finish the projec...",74eb249bbf


In [23]:
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [26]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/139 [00:00<?, ?it/s]

In [28]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

695

In [29]:
# calculate total cost - manual version

from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.11046824999999998

In [30]:
# calculate total cost - shortcut version

from evaluation_utils import calc_total_price

calc_total_price(usages)

0.11046824999999998

In [32]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [33]:
df_ground_truth.head(10)

,question,document
0,I just found this course — is it too late to j...,74eb249bbf
1,If I join after the course has already started...,74eb249bbf
2,What do I need to do to be eligible for the ce...,74eb249bbf
3,"Are late joiners allowed in the course, or do ...",74eb249bbf
4,"Can I still take part in the course now, and h...",74eb249bbf
5,Do I actually need a confirmation email to sta...,977bf7786c
6,"If I registered for the course, should I still...",977bf7786c
7,Is there a list of approved students for the L...,977bf7786c
8,What is the point of registration if I can alr...,977bf7786c
9,I signed up for the LLM Zoomcamp but haven’t g...,977bf7786c


In [34]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)